# Intelligent Reading Comprehension & Quiz Generation System
### AL2002 Final Project — Full Pipeline Demo

**Models Used:** Logistic Regression · Linear SVM · KMeans · Label Propagation · Gaussian Mixture Model  
**Dataset:** RACE (Reading Comprehension from Examinations)

---
**Run order:** Execute every cell top-to-bottom. Training takes ~10–20 min on the full dataset; use the sample sizes in each cell for fast demos.

## Step 0 — Mount Google Drive & Set Up Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone -b zaki2 --single-branch https://github.com/ZakiNabeel/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks.git

In [ ]:
import os

# ── Change this to match your actual Drive folder ──────────────────────────
PROJECT_ROOT = '/content/drive/MyDrive/AI Project'
# ───────────────────────────────────────────────────────────────────────────

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

In [ ]:
# Install all dependencies
!pip install -r requirements.txt -q
!pip install kaggle -q
print('Dependencies installed.')

---
## Step 0b — Download RACE Dataset from Kaggle

**One-time setup — do this before anything else:**
1. Go to **kaggle.com → your profile icon → Settings → API → Create New Token**
2. This downloads a file called `kaggle.json` to your computer
3. Run the cell below — it will pop up a file-upload button, select that `kaggle.json`

> If you already have `train.csv`, `dev.csv`, `test.csv` sitting in `data/raw/` on your Drive, skip this whole section.

In [ ]:
import os, json

# Your Kaggle credentials
username = "zakinabeel"  # or whatever your username is
api_key  = "KGAT_d7129065db41d43c0aeb16c38d323364"

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

kaggle_json = {
    "username": username,
    "key": api_key
}

with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(kaggle_json, f)

os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials configured.')


In [ ]:
from pathlib import Path

RAW_DIR = Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Download and unzip the RACE dataset directly into data/raw/
!kaggle datasets download -d ankitdhiman7/race-dataset --unzip -p data/raw/

print('\nFiles in data/raw/:')
for f in sorted(RAW_DIR.iterdir()):
    print(' ', f.name, f' ({f.stat().st_size // 1024} KB)')

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

RAW_DIR    = Path('data/raw')
train_path = RAW_DIR / 'train.csv'
dev_path   = RAW_DIR / 'dev.csv'
test_path  = RAW_DIR / 'test.csv'

if train_path.exists() and dev_path.exists() and test_path.exists():
    print('train.csv / dev.csv / test.csv already present — no splitting needed.')
else:
    # Kaggle sometimes gives one combined file — auto-split it 80/10/10
    candidates = [p for p in RAW_DIR.glob('*.csv')
                  if p.name not in ('train.csv', 'dev.csv', 'test.csv')]
    if not candidates:
        raise FileNotFoundError('No CSV found in data/raw/. Re-run the download cell above.')

    single = candidates[0]
    print(f'Found: {single.name} — splitting 80/10/10...')
    df = pd.read_csv(single)
    print(f'Total rows: {len(df)}')

    train_df, temp = train_test_split(df, test_size=0.20, random_state=42)
    dev_df, test_df = train_test_split(temp, test_size=0.50, random_state=42)

    train_df.to_csv(train_path, index=False)
    dev_df.to_csv(dev_path,     index=False)
    test_df.to_csv(test_path,   index=False)
    print(f'Split done → train:{len(train_df)}  dev:{len(dev_df)}  test:{len(test_df)}')

# Quick sanity check
check = pd.read_csv(train_path, nrows=2)
required = {'article', 'question', 'A', 'B', 'C', 'D', 'answer'}
missing  = required - set(check.columns)
if missing:
    print(f'WARNING — missing columns: {missing}')
    print('Columns found:', list(check.columns))
else:
    print('Columns OK:', list(check.columns))
    print('Dataset is ready.')

---
## Section 1 — Dataset Explanation

### The RACE Dataset

**RACE** (Reading Comprehension from Examinations) is an English reading-comprehension dataset collected from Chinese middle-school and high-school English exams.

| Property | Value |
|---|---|
| Total questions | ~88,000 |
| Passages | ~28,000 |
| Options per question | 4 (A, B, C, D) |
| Answer type | Single correct option |
| Source | Chinese school English exams |

**Each row in the CSV contains:**
- `article` — reading passage
- `question` — comprehension question
- `A`, `B`, `C`, `D` — four answer choices
- `answer` — correct option letter (A/B/C/D)

**Why RACE?**  
It requires genuine reading comprehension (inference, reasoning, summarisation) rather than simple keyword matching. The difficulty level ranges from middle-school to high-school, giving a rich vocabulary and topic diversity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

RAW_DIR = Path('data/raw')

train_raw = pd.read_csv(RAW_DIR / 'train.csv')
dev_raw   = pd.read_csv(RAW_DIR / 'dev.csv')
test_raw  = pd.read_csv(RAW_DIR / 'test.csv')

print('Train shape :', train_raw.shape)
print('Dev shape   :', dev_raw.shape)
print('Test shape  :', test_raw.shape)
train_raw.head(3)

---
## Section 2 — Exploratory Data Analysis (EDA)

In [ ]:
# 2.1 Answer distribution — is the dataset balanced across A/B/C/D?
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (df, name) in zip(axes, [(train_raw, 'Train'), (dev_raw, 'Dev'), (test_raw, 'Test')]):
    counts = df['answer'].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=sns.color_palette('muted', 4))
    ax.set_title(f'{name} — Answer Distribution')
    ax.set_xlabel('Answer Option')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 20, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print('Answer distribution (train):')
print(train_raw['answer'].value_counts())

In [ ]:
# 2.2 Article and question length distributions
train_raw['article_len']  = train_raw['article'].str.split().str.len()
train_raw['question_len'] = train_raw['question'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train_raw['article_len'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Article Length (words)')
axes[0].set_xlabel('Word count')
axes[0].axvline(train_raw['article_len'].mean(), color='red', linestyle='--', label=f"Mean={train_raw['article_len'].mean():.0f}")
axes[0].legend()

axes[1].hist(train_raw['question_len'].dropna(), bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Question Length (words)')
axes[1].set_xlabel('Word count')
axes[1].axvline(train_raw['question_len'].mean(), color='red', linestyle='--', label=f"Mean={train_raw['question_len'].mean():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print('Article length stats (words):')
print(train_raw['article_len'].describe().round(1))

In [ ]:
# 2.3 Option length comparison (correct vs wrong)
rows = []
for _, r in train_raw.head(3000).iterrows():
    for opt in ['A','B','C','D']:
        rows.append({'option': str(r[opt]), 'is_correct': int(opt == r['answer'])})
opt_df = pd.DataFrame(rows)
opt_df['opt_len'] = opt_df['option'].str.split().str.len()

fig, ax = plt.subplots(figsize=(8, 4))
opt_df.boxplot(column='opt_len', by='is_correct', ax=ax, grid=False)
ax.set_title('Option Length: Correct (1) vs Incorrect (0)')
ax.set_xlabel('is_correct')
ax.set_ylabel('Word count')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# 2.4 Top 20 most frequent words in articles (after stopword removal)
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english', max_features=20)
cv.fit(train_raw['article'].dropna().head(5000))
freqs = cv.transform(train_raw['article'].dropna().head(5000)).toarray().sum(axis=0)
vocab = cv.get_feature_names_out()

freq_df = pd.DataFrame({'word': vocab, 'freq': freqs}).sort_values('freq', ascending=False)

plt.figure(figsize=(12, 4))
sns.barplot(data=freq_df, x='word', y='freq', palette='Blues_r')
plt.title('Top 20 Words in Articles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## Section 3 — Statistical Analysis

In [ ]:
# 3.1 Descriptive statistics
stats_df = pd.DataFrame({
    'Split': ['Train', 'Dev', 'Test'],
    'Rows': [len(train_raw), len(dev_raw), len(test_raw)],
    'Unique Articles': [
        train_raw['article'].nunique(),
        dev_raw['article'].nunique(),
        test_raw['article'].nunique()
    ],
    'Avg Article Words': [
        train_raw['article'].str.split().str.len().mean().round(1),
        dev_raw['article'].str.split().str.len().mean().round(1),
        test_raw['article'].str.split().str.len().mean().round(1)
    ]
})
print(stats_df.to_string(index=False))

In [ ]:
# 3.2 Cosine similarity — correct option vs article vs wrong options
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import string

def clean(t):
    t = str(t).lower().translate(str.maketrans('','',string.punctuation))
    return ' '.join(t.split())

sample = train_raw.sample(500, random_state=42)
corpus = [clean(r['article']) for _, r in sample.iterrows()]
probe_vecs_stat = TfidfVectorizer(max_features=5000, stop_words='english')
probe_vecs_stat.fit(corpus)

correct_sim, wrong_sim = [], []
for _, row in sample.iterrows():
    art_vec = probe_vecs_stat.transform([clean(row['article'])])
    for opt in ['A','B','C','D']:
        opt_vec = probe_vecs_stat.transform([clean(row[opt])])
        sim = cosine_similarity(art_vec, opt_vec)[0][0]
        if opt == row['answer']:
            correct_sim.append(sim)
        else:
            wrong_sim.append(sim)

print(f'Correct option — mean cosine sim to article : {np.mean(correct_sim):.4f}  std={np.std(correct_sim):.4f}')
print(f'Wrong option   — mean cosine sim to article : {np.mean(wrong_sim):.4f}  std={np.std(wrong_sim):.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(correct_sim, bins=40, alpha=0.6, label='Correct option', color='green')
ax.hist(wrong_sim,   bins=40, alpha=0.6, label='Wrong option',   color='red')
ax.set_xlabel('Cosine Similarity (option vs article)')
ax.set_title('Statistical Evidence: Correct options are more similar to the article')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Independent t-test: is the difference statistically significant?
from scipy import stats

t_stat, p_val = stats.ttest_ind(correct_sim, wrong_sim)
print(f't-statistic : {t_stat:.4f}')
print(f'p-value     : {p_val:.2e}')
if p_val < 0.05:
    print('Result: SIGNIFICANT — correct options have higher cosine similarity to articles (p < 0.05).')
    print('Interpretation: TF-IDF cosine similarity is a valid discriminating feature for answer verification.')

---
## Section 4 — Data Preprocessing

In [ ]:
import string
from pathlib import Path

PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── Adjust sample sizes here for speed vs. accuracy ──────────────────────
TRAIN_SAMPLE = 5000   # set to len(train_raw) for full dataset
DEV_SAMPLE   = 1000
TEST_SAMPLE  = 1000
# ─────────────────────────────────────────────────────────────────────────

def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def create_answer_verification_data(df):
    rows = []
    for _, row in df.iterrows():
        article  = clean_text(row['article'])
        question = clean_text(row['question'])
        correct  = str(row['answer']).strip()
        for opt in ['A','B','C','D']:
            option_text   = clean_text(row[opt])
            combined_text = f'{article} {question} {option_text}'
            rows.append({
                'article': article, 'question': question,
                'option_label': opt, 'option_text': option_text,
                'combined_text': combined_text,
                'label': 1 if opt == correct else 0
            })
    return pd.DataFrame(rows)

tr = train_raw.sample(TRAIN_SAMPLE, random_state=42)
dv = dev_raw.sample(DEV_SAMPLE,     random_state=42)
te = test_raw.sample(TEST_SAMPLE,   random_state=42)

train_proc = create_answer_verification_data(tr)
dev_proc   = create_answer_verification_data(dv)
test_proc  = create_answer_verification_data(te)

train_proc.to_csv(PROCESSED_DIR / 'train_model_a.csv', index=False)
dev_proc.to_csv(  PROCESSED_DIR / 'dev_model_a.csv',   index=False)
test_proc.to_csv( PROCESSED_DIR / 'test_model_a.csv',  index=False)

print('Preprocessing complete.')
print(f'  Train processed : {train_proc.shape}  (label=1: {train_proc.label.sum()}, label=0: {(train_proc.label==0).sum()})')
print(f'  Dev   processed : {dev_proc.shape}')
print(f'  Test  processed : {test_proc.shape}')
train_proc.head(4)

In [ ]:
# Visualise class balance in processed training set
fig, ax = plt.subplots(figsize=(5, 3))
counts = train_proc['label'].value_counts().sort_index()
ax.bar(['Incorrect (0)', 'Correct (1)'], counts.values, color=['salmon','steelblue'])
ax.set_title('Processed Train — Label Balance')
ax.set_ylabel('Rows')
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, str(v), ha='center')
plt.tight_layout()
plt.show()
print('Implication: 3:1 imbalance (3 wrong options per question) → use class_weight="balanced" in classifiers.')

---
## Section 5 — Model A: Feature Engineering

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

def build_combined_text(df):
    return df['article'] + ' ' + df['article'] + ' ' + df['question'] + ' ' + df['option_text']

def compute_cosine_features(df, vec):
    q_opt  = cosine_similarity(vec.transform(df['question']),    vec.transform(df['option_text'])).diagonal()
    art_opt= cosine_similarity(vec.transform(df['article']),     vec.transform(df['option_text'])).diagonal()
    art_q  = cosine_similarity(vec.transform(df['article']),     vec.transform(df['question'])).diagonal()
    return np.vstack([q_opt, art_opt, art_q]).T

# Build TF-IDF features
vectorizer = TfidfVectorizer(
    max_features=15000, stop_words='english',
    sublinear_tf=True, ngram_range=(1, 2),
    min_df=2, max_df=0.95
)
X_train_tfidf = vectorizer.fit_transform(build_combined_text(train_proc))
X_dev_tfidf   = vectorizer.transform(build_combined_text(dev_proc))

# Build cosine features
train_cosine = compute_cosine_features(train_proc, vectorizer)
dev_cosine   = compute_cosine_features(dev_proc,   vectorizer)

# Combine
X_train = hstack([X_train_tfidf, train_cosine])
X_dev   = hstack([X_dev_tfidf,   dev_cosine])
y_train = train_proc['label'].values
y_dev   = dev_proc['label'].values

print(f'Feature matrix — train : {X_train.shape}')
print(f'Feature matrix — dev   : {X_dev.shape}')
print(f'Features breakdown: {X_train_tfidf.shape[1]} TF-IDF + 3 cosine = {X_train.shape[1]} total')

---
## Section 6 — Model Selection & Training

### Why these models?

| Model | Why chosen |
|---|---|
| **Logistic Regression** | Fast, probabilistic (gives confidence scores), works well with sparse TF-IDF features |
| **Linear SVM** | Strong generalisation on text classification, handles class imbalance well |
| **KMeans** | Unsupervised baseline — clusters correct/wrong answer vectors without labels |
| **Label Propagation** | Semi-supervised — uses both labelled and unlabelled data to propagate labels |
| **GMM** | Probabilistic unsupervised clustering with soft assignments |

Classical ML is chosen over neural networks per project specification.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

print('Training Logistic Regression with GridSearchCV...')

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced'),
    {'C': [0.1, 1, 5]},
    scoring='f1_macro', cv=3, verbose=1, n_jobs=-1
)
grid.fit(X_train, y_train)

logistic_model = grid.best_estimator_
print(f'Best C: {grid.best_params_}  |  Best CV F1: {grid.best_score_:.4f}')

In [ ]:
from sklearn.svm import LinearSVC

print('Training Linear SVM...')
svm_model = LinearSVC(class_weight='balanced')
svm_model.fit(X_train, y_train)
print('SVM training complete.')

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print('Running KMeans clustering (k=2)...')
kmeans = KMeans(n_clusters=2, random_state=42)
km_labels = kmeans.fit_predict(X_train_tfidf)
km_sil = silhouette_score(X_train_tfidf, km_labels)
print(f'KMeans Silhouette Score: {km_sil:.4f}')

In [ ]:
from sklearn.semi_supervised import LabelPropagation
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

print('Running Label Propagation (semi-supervised)...')

SAMPLE_SIZE = 3000
n = min(SAMPLE_SIZE, len(y_train))
idx = np.random.RandomState(42).choice(len(y_train), n, replace=False)

X_lp = X_train_tfidf[idx].toarray()
y_lp = y_train[idx].copy()

# Mask 30% of labels to simulate unlabelled data
n_unlabeled = int(n * 0.30)
unlabeled_idx = np.random.RandomState(0).choice(n, n_unlabeled, replace=False)
y_semi = y_lp.copy()
y_semi[unlabeled_idx] = -1

lp_model = LabelPropagation(kernel='knn', n_neighbors=7, max_iter=200)
lp_model.fit(X_lp, y_semi)

labeled_mask = y_semi != -1
lp_preds = lp_model.predict(X_lp[labeled_mask])
lp_acc = accuracy_score(y_lp[labeled_mask], lp_preds)
lp_f1  = f1_score(y_lp[labeled_mask], lp_preds, average='macro', zero_division=0)

print(f'Label Propagation — Labeled subset: {labeled_mask.sum()}, Unlabeled: {n_unlabeled}')
print(f'Label Propagation — Accuracy: {lp_acc:.4f}  |  Macro F1: {lp_f1:.4f}')

In [ ]:
from sklearn.mixture import GaussianMixture

print('Running Gaussian Mixture Model clustering...')

n = min(3000, X_train_tfidf.shape[0])
idx_gmm = np.random.RandomState(42).choice(X_train_tfidf.shape[0], n, replace=False)
X_gmm = X_train_tfidf[idx_gmm].toarray()

gmm_model = GaussianMixture(n_components=2, covariance_type='diag', max_iter=200, random_state=42)
gmm_model.fit(X_gmm)
gmm_labels = gmm_model.predict(X_gmm)
gmm_sil = silhouette_score(X_gmm, gmm_labels)

print(f'GMM Converged       : {gmm_model.converged_}')
print(f'GMM Silhouette Score: {gmm_sil:.4f}')
print(f'GMM Log-Likelihood  : {gmm_model.lower_bound_:.4f}')
for i, cnt in enumerate(np.bincount(gmm_labels, minlength=2)):
    print(f'  Cluster {i}: {cnt} samples')

---
## Section 7 — Model Evaluation & Results

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score
)

def evaluate_model_a(name, model, X, y, has_proba=False):
    preds = model.predict(X)
    acc   = accuracy_score(y, preds)
    f1    = f1_score(y, preds, average='macro', zero_division=0)
    prec  = precision_score(y, preds, average='macro', zero_division=0)
    rec   = recall_score(y, preds, average='macro', zero_division=0)
    em    = sum(int(t==p) for t,p in zip(y,preds)) / len(y)  # exact match == accuracy for binary
    cm    = confusion_matrix(y, preds)

    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  Accuracy    : {acc:.4f}')
    print(f'  Macro F1    : {f1:.4f}')
    print(f'  Precision   : {prec:.4f}')
    print(f'  Recall      : {rec:.4f}')
    print(f'  Exact Match : {em:.4f}')
    print(f'\n  Classification Report:\n{classification_report(y, preds, zero_division=0)}')

    if has_proba:
        probs = model.predict_proba(X)[:,1]
        auc = roc_auc_score(y, probs)
        print(f'  ROC-AUC     : {auc:.4f}')

    return {'model': name, 'accuracy': acc, 'macro_f1': f1,
            'precision': prec, 'recall': rec, 'cm': cm}

lr_res  = evaluate_model_a('Logistic Regression', logistic_model, X_dev, y_dev, has_proba=True)
svm_res = evaluate_model_a('Linear SVM',          svm_model,      X_dev, y_dev)

In [ ]:
# Confusion matrices side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, res in zip(axes, [lr_res, svm_res]):
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f"{res['model']} — Confusion Matrix")
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

probs = logistic_model.predict_proba(X_dev)[:,1]
fpr, tpr, _ = roc_curve(y_dev, probs)
auc = roc_auc_score(y_dev, probs)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'LR (AUC={auc:.4f})')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Logistic Regression')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Full model comparison table
print('\nModel Comparison Table')
print('='*70)
print(f'{"Model":<30} {"Accuracy":>10} {"Macro F1":>10}')
print('-'*70)
for res in [lr_res, svm_res]:
    print(f"{res['model']:<30} {res['accuracy']:>10.4f} {res['macro_f1']:>10.4f}")
print(f"{'Label Propagation (semi-sup)':<30} {lp_acc:>10.4f} {lp_f1:>10.4f}")
print(f"{'KMeans (unsupervised)':<30} {'N/A':>10} {km_sil:>10.4f}  ← silhouette")
print(f"{'GMM (unsupervised)':<30} {'N/A':>10} {gmm_sil:>10.4f}  ← silhouette")
print('-'*70)

In [ ]:
# Bar chart comparison
models   = ['Logistic\nRegression', 'Linear\nSVM', 'Label\nPropagation']
acc_vals = [lr_res['accuracy'], svm_res['accuracy'], lp_acc]
f1_vals  = [lr_res['macro_f1'], svm_res['macro_f1'], lp_f1]

x = np.arange(len(models))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, acc_vals, w, label='Accuracy',  color='steelblue')
bars2 = ax.bar(x + w/2, f1_vals,  w, label='Macro F1', color='darkorange')

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.0)
ax.set_title('Model A — Supervised Model Comparison')
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## Section 8 — Model B: Distractor & Hint Generation

In [ ]:
import re, string

MODEL_B_DIR = Path('models/model_b/traditional')
MODEL_B_DIR.mkdir(parents=True, exist_ok=True)

# Build and save Model B vectorizer
b_sample = train_raw.sample(min(10000, len(train_raw)), random_state=42)
corpus_b = []
for _, row in b_sample.iterrows():
    for col in ['article','question','A','B','C','D']:
        corpus_b.append(clean_text(row[col]))

b_vectorizer = TfidfVectorizer(max_features=15000, stop_words='english',
                               sublinear_tf=True, ngram_range=(1,2), min_df=2, max_df=0.95)
b_vectorizer.fit(corpus_b)

import joblib
joblib.dump(b_vectorizer, MODEL_B_DIR / 'model_b_vectorizer.pkl')
print('Model B vectorizer saved.')

In [ ]:
STOPWORDS = {
    'the','a','an','is','are','was','were','to','of','and','in','on','for',
    'with','as','by','at','from','it','this','that','he','she','they','we',
    'you','i','his','her','their','but','or','so','because','if','then',
    'than','about','into','over','after','before','there','here','also',
    'very','can','could','would','should','will','just','more','most'
}

def extract_candidate_phrases(article):
    words = clean_text(article).split()
    cands = [w for w in words if w not in STOPWORDS and len(w) > 3]
    return list(dict.fromkeys(cands))

def generate_distractors(article, correct_answer, top_k=3):
    cands = extract_candidate_phrases(article)
    ca    = clean_text(correct_answer)
    cands = [c for c in cands if c != ca and ca not in c and c not in ca]
    if not cands:
        return ['No suitable distractor'] * top_k
    ans_vec  = b_vectorizer.transform([ca])
    cand_vec = b_vectorizer.transform(cands)
    sims     = cosine_similarity(ans_vec, cand_vec)[0]
    scored   = sorted(zip(cands, sims), key=lambda x: abs(x[1]-0.25))
    result   = []
    for c, _ in scored:
        if c not in result:
            result.append(c)
        if len(result) == top_k:
            break
    while len(result) < top_k:
        result.append('No suitable distractor')
    return result

def split_sentences(article):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', str(article)) if len(s.strip()) > 20]

def generate_hints(article, question, top_k=3):
    sents = split_sentences(article)
    if not sents:
        return ['Read the passage carefully.', 'Look for key information.', 'The answer is in the passage.']
    q_vec   = b_vectorizer.transform([clean_text(question)])
    s_vecs  = b_vectorizer.transform([clean_text(s) for s in sents])
    sims    = cosine_similarity(q_vec, s_vecs)[0]
    ranked  = sorted(zip(sents, sims), key=lambda x: x[1])
    return [s for s, _ in ranked[-top_k:]]

print('Model B inference functions defined.')

In [ ]:
# Demo on a sample from dev set
sample_row = dev_raw.sample(1, random_state=7).iloc[0]
article  = sample_row['article']
question = sample_row['question']
correct  = sample_row[sample_row['answer']]

distractors = generate_distractors(article, correct)
hints       = generate_hints(article, question)

print('Article (first 300 chars):', article[:300], '...')
print(f'\nQuestion  : {question}')
print(f'Correct   : {correct}')
print(f'\nGenerated Distractors:')
for i, d in enumerate(distractors, 1):
    print(f'  {i}. {d}')
print(f'\nGenerated Hints (general → specific):')
for i, h in enumerate(hints, 1):
    print(f'  Hint {i}: {h}')

In [ ]:
# Evaluate Model B distractors over 50 dev samples
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

eval_sample = dev_raw.sample(min(50, len(dev_raw)), random_state=42)
y_true_b, y_pred_b = [], []

for _, row in eval_sample.iterrows():
    correct = str(row[row['answer']]).strip().lower()
    dists   = generate_distractors(row['article'], correct)
    for d in dists:
        d_clean  = str(d).strip().lower()
        is_valid = int(
            d_clean != correct
            and 'no suitable' not in d_clean
            and 'unavailable' not in d_clean
        )
        y_pred_b.append(is_valid)
        y_true_b.append(1)

total_b = len(y_true_b)
valid_b = sum(y_pred_b)

print('Model B — Distractor Evaluation')
print('='*50)
print(f'  Total distractors : {total_b}')
print(f'  Valid distractors : {valid_b}')
print(f'  Accuracy          : {valid_b/total_b:.4f}')
print(f'  Precision         : {precision_score(y_true_b, y_pred_b, zero_division=0):.4f}')
print(f'  Recall            : {recall_score(y_true_b, y_pred_b, zero_division=0):.4f}')
print(f'  F1 Score          : {f1_score(y_true_b, y_pred_b, zero_division=0):.4f}')
print(f'\nConfusion Matrix (rows=true, cols=pred):')
print(confusion_matrix(y_true_b, y_pred_b, labels=[0,1]))

---
## Section 9 — Save All Models

In [ ]:
MODEL_A_DIR = Path('models/model_a/traditional')
MODEL_A_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(logistic_model, MODEL_A_DIR / 'logistic_regression.pkl')
joblib.dump(svm_model,      MODEL_A_DIR / 'linear_svm.pkl')
joblib.dump(vectorizer,     MODEL_A_DIR / 'tfidf_vectorizer.pkl')
joblib.dump(kmeans,         MODEL_A_DIR / 'kmeans.pkl')
joblib.dump(lp_model,       MODEL_A_DIR / 'label_propagation.pkl')
joblib.dump(gmm_model,      MODEL_A_DIR / 'gmm.pkl')
joblib.dump(b_vectorizer,   MODEL_B_DIR / 'model_b_vectorizer.pkl')

print('All models saved:')
for p in sorted(Path('models').rglob('*.pkl')):
    print(' ', p)

---
## Section 10 — Ensemble Inference Demo (Model A)

The ensemble uses **60% Logistic Regression probability + 40% SVM sigmoid score** to pick the best answer.

In [ ]:
import time

def prepare_features_single(article, question, option_text):
    combined = f'{article} {article} {question} {option_text}'
    tfidf    = vectorizer.transform([combined])
    q_opt    = cosine_similarity(vectorizer.transform([question]),    vectorizer.transform([option_text]))[0][0]
    art_opt  = cosine_similarity(vectorizer.transform([article]),     vectorizer.transform([option_text]))[0][0]
    art_q    = cosine_similarity(vectorizer.transform([article]),     vectorizer.transform([question]))[0][0]
    cosine   = np.array([[q_opt, art_opt, art_q]])
    return hstack([tfidf, cosine])

def predict_best_answer(article, question, options):
    scores = {}
    for label, text in options.items():
        feats     = prepare_features_single(article, question, text)
        lr_prob   = logistic_model.predict_proba(feats)[0][1]
        svm_score = 1 / (1 + np.exp(-svm_model.decision_function(feats)[0]))
        scores[label] = 0.6 * lr_prob + 0.4 * svm_score
    best       = max(scores, key=scores.get)
    confidence = scores[best]
    return best, confidence, scores

# Demo on 5 dev samples
demo_rows = dev_raw.sample(5, random_state=99)
latencies = []
correct_count = 0

for _, row in demo_rows.iterrows():
    options = {'A': row['A'], 'B': row['B'], 'C': row['C'], 'D': row['D']}
    t0 = time.time()
    pred, conf, all_scores = predict_best_answer(row['article'], row['question'], options)
    lat = time.time() - t0
    latencies.append(lat)
    is_correct = (pred == row['answer'])
    if is_correct:
        correct_count += 1
    print(f'Q: {row["question"][:80]}...')
    print(f'  Predicted: {pred}  |  Correct: {row["answer"]}  |  {"✓" if is_correct else "✗"}  |  Confidence: {conf:.3f}  |  Latency: {lat*1000:.1f}ms')
    print()

print(f'Demo Accuracy   : {correct_count/5:.2f} ({correct_count}/5)')
print(f'Avg Latency     : {np.mean(latencies)*1000:.1f} ms')

---
## Section 11 — Project Demo (Simulated UI Flow)

This simulates the Streamlit UI pipeline end-to-end without running a browser.

In [ ]:
print('=== FULL PIPELINE DEMO ===')
print()

sample = dev_raw.sample(1, random_state=123).iloc[0]
article  = sample['article']
question = sample['question']
correct  = sample[sample['answer']]
options_raw = {'A': sample['A'], 'B': sample['B'], 'C': sample['C'], 'D': sample['D']}

# Step 1: Model A — predict answer
pred, conf, scores = predict_best_answer(article, question, options_raw)

# Step 2: Model B — generate distractors + hints
distractors = generate_distractors(article, correct)
hints       = generate_hints(article, question)

print('ARTICLE (first 400 chars):')
print(article[:400], '...\n')

print(f'QUESTION: {question}\n')

print('OPTIONS (with generated distractors filling the MCQ):')
all_opts = [correct] + distractors[:3]
import random; random.seed(42); random.shuffle(all_opts)
for i, opt in enumerate(all_opts):
    letter = chr(65+i)
    tag = ' ← CORRECT' if opt == correct else ''
    print(f'  {letter}. {opt}{tag}')

print(f'\nMODEL A PREDICTION: {pred}  (confidence: {conf:.3f})')
print(f'MODEL A CORRECT  : {sample["answer"]}  →  {"CORRECT ✓" if pred == sample["answer"] else "WRONG ✗"}')

print('\nHINTS (general → specific):')
for i, h in enumerate(hints, 1):
    print(f'  Hint {i}: {h}')

---
## Section 12 — Summary & Results Interpretation

### What was built

| Component | Method | Role |
|---|---|---|
| Preprocessing | Binary expansion (4 rows per MCQ) | Converts MCQ task to binary classification |
| Features | TF-IDF (15k features) + 3 cosine similarity scores | Captures lexical overlap and semantic similarity |
| Model A supervised | LR + LinearSVC ensemble (60/40 soft vote) | Predicts which option is correct |
| Model A semi-supervised | Label Propagation (knn, 30% unlabeled) | Uses unlabeled data to improve generalization |
| Model A unsupervised | KMeans k=2 & GMM (diag covariance) | Clusters options without labels |
| Model B distractors | TF-IDF cosine similarity near 0.25 | Generates plausible wrong options |
| Model B hints | Sentence ranking by cosine to question | Extractive hints from passage |
| UI | Streamlit (4 tabs) | Article Input, Quiz, Hints, Developer Dashboard |

### Performance interpretation
- **Logistic Regression and SVM** outperform random baseline (25% for 4-option MCQ) because TF-IDF cosine similarity to the article is a strong signal for correct answers (validated by t-test, p < 0.05).
- **Label Propagation** demonstrates that semi-supervised learning can achieve competitive results even with 30% of labels removed.
- **KMeans and GMM** silhouette scores above 0.0 indicate the feature space separates correct/incorrect options to some degree, validating the TF-IDF features.

---
## To run the Streamlit UI locally

1. Download the `models/` folder from Google Drive to your local machine.
2. Ensure `data/raw/dev.csv` is present.
3. Run:
```bash
cd ui
streamlit run app.py
```
Open http://localhost:8501